# CNN for Classifying EMG Muscle Contractions 

In [1]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle


from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

matplotlib.use('QtAgg') 
mne.set_log_level("CRITICAL")

# Defining initial variables 

In [2]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']
subject_exclude = ["NL01SS", "NL02IF",
                   "NL05WW01", "RL12JL03", "RL07BR02", 
                   "RL11JH"] # trials to exclude based on trigger count 


inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/Consciousness_Research/Python_Scripts/EEG_data"
current_index=0
inter_trigger_length=10
window = 50 
step = 1 

In [3]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers 

In [4]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     global frq
     global window 
     global step 

     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  
     mav = np.mean(np.abs(epoch_sw), axis=1)
     mavs = np.diff(mav)

     ''' 
     # frequency features 
    
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]
     '''

  
     return var, rms, wl, mavs


Creating Dataframe

In [6]:
# load in features
#features_all = pd.read_excel("training_features_11032026.xlsx")
features_all = pd.read_pickle("training_features_11032026.pkl")

# CNN 

Defining Model 

In [ ]:
def CNN_model(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) #, kernel_regularizer=l2(0.001)))  # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [ ]:
def one_subject_test(subj_omit,model):

    idx = features_all.index[features_all["Subject"] == subj_omit][0]
    acc_corr = 0 
    acc_zygo = 0 

    for t in range(idx,idx+60):
        subject_info = features_all.loc[t,["Subject"]].iloc[0]
        nap_info = features_all.loc[t,["Nap Number"]].iloc[0]
        print(subject_info,nap_info)


        sample_corr = features_all.loc[t,["Corr"]]
        sample_corr = sample_corr.to_numpy()
        sample_corr = np.array(sample_corr.tolist()).reshape(1, 2251,1)

        sample_zygo = features_all.loc[t,["Zygo"]]
        sample_zygo = sample_zygo.to_numpy()
        sample_zygo = np.array(sample_zygo.tolist()).reshape(1, 2251,1)

        prediction_corr = model.predict(sample_corr, verbose=0)
        prediction_zygo = model.predict(sample_zygo, verbose=0)

        prediction_corr_contract = np.argmax(prediction_corr, axis=1)[0]
        prediction_zygo_contract = np.argmax(prediction_zygo, axis=1)[0]

        actual_corr_contract = 
        actual_zygo_contract = 

        if (prediction_corr_contract == actual_corr_contract):
            acc_corr += 1 
        if (prediction_zygo_contract == actual_zygo_contract):
            acc_zygo += 1 


 
    return [acc_corr/60, acc_zygo/60] # returns percent trials correct 
      

In [ ]:
def plot_accuracy(model_history,i):
    
    plt.plot(model_history.history['accuracy'], label='accuracy')
    plt.plot(model_history.history['val_accuracy'], label = 'val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.ylim([0.5, 1])
    plt.title(f"Fold {i}")
    plt.legend(loc='lower right')

Training with KFolds 

In [ ]:
subjects = list(set(features_all["Subject"])) 
cv_scores_all = [] 
results_all = []

In [ ]:
# loop over all subjects 
subjects = list(set(features_all["Subject"])) 

for subj_omit in subjects:
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]



    X_zygo = features_all_temp[["Zygo"]] 
    X_corr = features_all_temp[["Corr"]]

    X_zygo = X_zygo.to_numpy()
    X_zygo = np.array(X_zygo.tolist())
    X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

    X_corr = X_corr.to_numpy()
    X_corr = np.array(X_corr.tolist())
    X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

    X = np.concatenate((X_zygo, X_corr), axis=0)
    y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                        features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

    
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # keep 20% purely for testing 

    input_shape = X_train_full.shape[1:]  # Determine input shape based on the processed data with  
    num_classes = len(np.unique(y))   # Determine the number of classes based on unique values in the target vector

    input_shape = (input_shape[0], 1)  # Convert input shape to (input_shape[0], 1)
    feature_num = np.shape(X_zygo)[2]
    epoch_len = np.shape(X_zygo)[1]

    # Create CNN model using the adjusted input shape and number of classes
    kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
    cvScores=[]

    k = 1
    for train_index, test_index in kf.split(X_train_full):
        print(f"Fold: {k} ==================================================================")
        
        X_train, X_val = X[train_index], X[test_index]
        y_train, y_val = y[train_index], y[test_index]

        model = CNN_model(input_shape, num_classes,feature_num)
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

        #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
        
        model_history_kfold = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
        #plot_accuracy(model_history_kfold,i)
        
        scores = model.evaluate(X_test,y_test)
        cvScores.append(scores[1] * 100)

        k += 1 
        


    model_history = model.fit(X_train_full, y_train_full, epochs=10, validation_data=(X_test, y_test)) #, callbacks=[early_stop])
    subj_accuracy =  one_subject_test(subj_omit,model)
    
    cv_scores_all.append(cvScores)
    results_all.append(subj_accuracy)

    # save model
    model_name =  "emg_omit_"+subj_omit+".pkl"
    with open(model_name, "wb") as f:
        pickle.dump(model, f)